# The receptive field of a CNN

_Deep Learning_

**How big a patch of the original image one deep output neuron actually sees.**

Pick one number in a deep feature map. Ask: which input pixels, if you changed them, would change this number? That set of input pixels is its receptive field.

       Each convolutional layer's filter looks at a small window, but that window sits on the previous layer's neurons — which themselves looked at windows of the layer before. So as you stack layers, the patch of the ORIGINAL input that one neuron depends on keeps growing.

---

This notebook is a practice scaffold for the **The receptive field of a CNN** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — NumPy

The receptive field is computed by a short recursion that walks layer by layer through the network. We define the helper, run it on a small stack, then sanity-check it against a known value.

### Step 1 — Define the receptive-field recursion

One output pixel starts by seeing just itself (`R = 1`). Each layer with filter size `F` widens that reach by `(F - 1)` input pixels — but scaled by `jump`, the cumulative spacing built up from all earlier strides. After adding the layer's contribution we update `jump` by multiplying in this layer's stride, so deeper layers reach proportionally further.

In [ ]:
import numpy as np

def receptive_field(layers):
    """layers: list of (filter_size F, stride S) per layer.
    Returns the receptive field after each layer using the recursion
        R <- R + (F - 1) * jump,   jump <- jump * S
    where 'jump' = product of all earlier strides (starts at 1, since S0 = 1).
    """
    R = 1          # one pixel sees itself
    jump = 1       # spacing between adjacent neurons, in input pixels (S0 = 1)
    out = []
    for (F, S) in layers:
        R = R + (F - 1) * jump   # this layer adds (F-1) scaled by the cumulative stride
        jump = jump * S          # later layers now step 'S' times wider
        out.append(R)
    return np.array(out)

### Step 2 — Run it on a small stack

We feed in three 3x3 stride-1 layers followed by a 3x3 stride-2 layer and print the receptive field after each. Notice how the stride-2 layer makes the field jump faster than the stride-1 layers before it — the earlier stride has widened `jump`.

In [ ]:
# real small stack: three 3x3 stride-1 layers, then a 3x3 stride-2 layer
stack = [(3, 1), (3, 1), (3, 1), (3, 2)]
rf = receptive_field(stack)

for k, ((F, S), R) in enumerate(zip(stack, rf), start=1):
    print("layer %d: %dx%d stride %d  ->  receptive field = %d" % (k, F, F, S, R))

### Step 3 — Sanity-check against a known value

A classic result is that two stacked 3x3 stride-1 layers see a 5x5 patch — the same as a single 5x5 filter. Running the helper on exactly that stack should print `5`, confirming the recursion is correct.

In [ ]:
# sanity check: cheat-sheet example F1=F2=3, S1=S2=1 gives R2 = 5
two_layer_rf = receptive_field([(3, 1), (3, 1)])[-1]
print("check two 3x3 stride-1 layers, R2 =", two_layer_rf)

## Visualize the data & results

_How fast does the receptive field grow with depth — for an all-stride-1 3x3 net versus one with a few stride-2 layers?_

### Step 1 — Redefine the recursion compactly

This cell stands alone, so we restate the same `receptive_field` helper. The logic is identical to before: start at one pixel, add `(F - 1) * jump` per layer, and grow `jump` by each layer's stride.

In [ ]:
import numpy as np

def receptive_field(layers):
    R, jump = 1, 1                 # R0 = 1 pixel, jump = product of earlier strides (S0 = 1)
    out = []
    for (F, S) in layers:
        R = R + (F - 1) * jump     # additive growth, scaled by cumulative stride
        jump = jump * S
        out.append(R)
    return np.array(out)

### Step 2 — Build two 12-layer architectures

To compare growth rates we define two deep stacks. `arch_a` is twelve plain 3x3 stride-1 layers. `arch_b` is the same depth but inserts a stride-2 layer at positions 3, 6, and 9. Those occasional strides are what make the second network's receptive field balloon.

In [ ]:
depth = 12
arch_a = [(3, 1)] * depth   # all 3x3 stride-1

# arch_b: same depth, but stride-2 at layers 3, 6, and 9
arch_b = []
for k in range(1, depth + 1):
    stride = 2 if k in (3, 6, 9) else 1
    arch_b.append((3, stride))

### Step 3 — Compare the growth

Running both architectures through the recursion and printing the per-layer receptive fields makes the contrast obvious: the all-stride-1 net grows **linearly** (3, 5, 7, …), while the strided net leaps ahead each time a stride-2 layer multiplies the jump.

In [ ]:
ra = receptive_field(arch_a)
rb = receptive_field(arch_b)
layers = np.arange(1, depth + 1)

print("layer :", layers.tolist())
print("all s1:", ra.tolist())     # [3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23, 25]
print("strided:", rb.tolist())    # [3, 5, 7, 11, 15, 19, 27, 35, 43, 59, 75, 91]

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You stack three $3\times3$ stride-1 convolutional layers. What is the receptive field at layer 3?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Each $3\times3$ stride-1 layer contributes $(F-1)\prod S_i = 2\cdot 1 = 2$. — _All strides are 1, so every product is 1; each layer adds $F-1 = 2$._
- Sum three contributions and add the starting pixel: $1 + 2 + 2 + 2$. — _The formula starts at 1 (a single pixel) and adds each layer's term._

**Answer:** $R_3 = 1 + 2 + 2 + 2 = 7$. Three stacked $3\times3$ stride-1 layers match a single $7\times7$ filter's receptive field.

</details>

**Problem 2.** Layer 1 is $3\times3$ stride 2; layer 2 is $3\times3$ stride 1. What is $R_2$, and why is it bigger than the all-stride-1 case ($R_2 = 5$)?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Layer 1: $(3-1)\prod_{i=0}^{0} S_i = 2 \cdot S_0 = 2 \cdot 1 = 2$. — _The product before layer 1 is empty, equal to $S_0 = 1$._
- Layer 2: $(3-1)\prod_{i=0}^{1} S_i = 2 \cdot (S_0 S_1) = 2 \cdot (1\cdot 2) = 4$. — _Layer 1's stride of 2 now multiplies layer 2's contribution._
- Add: $1 + 2 + 4$. — _Start at one pixel, add both layer terms._

**Answer:** $R_2 = 1 + 2 + 4 = 7$. The early stride of 2 doubled layer 2's reach, so the receptive field grew faster than the all-stride-1 value of 5.

</details>

**Problem 3.** Your deepest feature must see a $32\times32$ object, but it currently sees only a $17\times17$ patch. Name two cheap ways to grow the receptive field without adding many parameters.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Add a stride (or pooling) layer. — _A stride multiplies the contribution of all later layers, so the receptive field grows much faster per added layer._
- Use dilated (atrous) convolutions. — _Dilation spreads the same $F$ filter taps over a wider input span, enlarging $F-1$'s effective reach with no extra weights._

**Answer:** Insert a strided/pooling layer (multiplies later layers' growth) and/or switch some layers to dilated convolutions (wider reach at the same parameter count). Both grow the receptive field cheaply; adding plain stride-1 layers grows it only linearly.

</details>